# Retrieval-augmented generation: a hands-on implementation

**Track:** Search · **Guide:** [`047-rag.md`](../guides/concepts/047-rag.md)

RAG retrieves relevant evidence at answer time and gives that evidence to a generator to ground its response.

> **Intuition:** An open-book exam: first find the relevant pages, then write an answer that points back to them.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Retrieval-augmented generation]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Answer policy questions from versioned manuals, cite passages, and abstain if evidence is missing.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Build a source corpus

RAG starts with evidence-bearing passages and source metadata, not a free-form answer.


In [ ]:
documents = [
    {"id": "returns-v3#p2", "text": "Unused items can be returned within 30 days of delivery."},
    {"id": "shipping-v2#p1", "text": "Standard delivery usually takes 3 to 5 business days."},
    {"id": "warranty-v1#p4", "text": "The limited warranty lasts for one year."},
]
question = "How long is the return period?"

## 2. Retrieve evidence before answering

A simple lexical ranker keeps the lesson local and deterministic. Production retrieval should be tested for recall.


In [ ]:
def retrieve(question, documents, k=2):
    terms = set(question.lower().split())
    ranked = []
    for document in documents:
        overlap = len(terms & set(document["text"].lower().replace(".", "").split()))
        ranked.append((overlap, document))
    return [document for score, document in sorted(ranked, key=lambda x: -x[0]) if score][:k]

evidence = retrieve(question, documents)
for passage in evidence:
    print(passage["id"], "->", passage["text"])

## 3. Produce a grounded answer with citations

This extractive answer does not call an LLM: it returns a relevant source sentence. Abstain when retrieval yields no evidence.


In [ ]:
def grounded_answer(question, evidence):
    if not evidence:
        return {"answer": "I could not find supporting evidence.", "sources": []}
    return {
        "answer": evidence[0]["text"],
        "sources": [document["id"] for document in evidence],
    }

answer = grounded_answer(question, evidence)
print(answer)
assert answer["sources"]

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/047-rag.md) and the
[learning roadmap](../guides/ROADMAP.md).
